<img src="http://developer.download.nvidia.com/notebooks/dlsw-notebooks/tensorrt_torchtrt_efficientnet/nvidia_logo.png" width="90px">

# PySpark LLM Inference: MinerU-HTML Processing

In this notebook, we demonstrate distributed batch inference with [MinerU-HTML](https://github.com/opendatalab/MinerU-HTML/tree/main) framework and model.

In [1]:
import os

# Manually enable Huggingface tokenizer parallelism to avoid disabling with PySpark parallelism.
# See (https://github.com/huggingface/transformers/issues/5486) for more info. 
os.environ["TOKENIZERS_PARALLELISM"] = "true"

# vLLM does CUDA init at import time. Forking will try to re-initialize CUDA if vLLM was imported before and throw an error.
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

In [ ]:
from huggingface_hub import snapshot_download

MODEL_PATH = os.path.abspath("mineru-html")
snapshot_download(
    repo_id="opendatalab/MinerU-HTML",
    local_dir=MODEL_PATH
)

## Warmup: Running locally

**Note**: If the driver node does not have sufficient GPU capacity, proceed to the PySpark section.

In [3]:
from dripper.api import Dripper

dripper = Dripper(
    config={
        'model_path': MODEL_PATH,
        'vllm_kwargs': {
            'gpu_memory_utilization': 0.15,
        }
    }
)

In [ ]:
html_content = """
<html>
  <body>
    <div>
    <h1>This is a title</h1>
    <p>This is a paragraph</p>
    <p>This is another paragraph</p>
    </div>
    <div>
    <p>Related content</p>
    <p>Advertising content</p>
    </div>
  </body>
</html>
"""

result = dripper.process(html_content)

In [5]:
print(result[0].main_html)

<html><body>
<div>
<h1 _item_id="1">This is a title</h1>
<p _item_id="2">This is a paragraph</p>
<p _item_id="3">This is another paragraph</p>
</div>
</body></html>


Unload the model to free up the GPU for the PySpark section.

In [6]:
import contextlib
import gc
import torch
from vllm.distributed import destroy_model_parallel, destroy_distributed_environment

def cleanup():
    destroy_model_parallel()
    destroy_distributed_environment()
    with contextlib.suppress(AssertionError):
        torch.distributed.destroy_process_group()
    gc.collect()
    torch.cuda.empty_cache()

del dripper
cleanup()

[rank0]:[W1209 15:08:46.264724515 ProcessGroupNCCL.cpp:1524] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


## PySpark

In [3]:
import pandas as pd
from pyspark.sql.types import *
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, col, struct, length, lit, concat
from pyspark.ml.functions import predict_batch_udf

In [4]:
import os
import socket
import datasets
from datasets import load_dataset
datasets.disable_progress_bars()

In [5]:
MODEL_PATH = os.path.abspath("mineru-html")

#### Create Spark Session

In [6]:
conf = SparkConf()

conda_env = os.environ.get("CONDA_PREFIX")
hostname = socket.gethostname()
conf.setMaster(f"spark://{hostname}:7077")
conf.set("spark.pyspark.python", f"{conda_env}/bin/python")
conf.set("spark.pyspark.driver.python", f"{conda_env}/bin/python")
conf.set("spark.executor.cores", "8")
conf.set("spark.task.maxFailures", "1")
conf.set("spark.task.resource.gpu.amount", "0.125")
conf.set("spark.executor.resource.gpu.amount", "1")
conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
conf.set("spark.python.worker.reuse", "true")

spark = SparkSession.builder.appName("spark-dl-examples").config(conf=conf).getOrCreate()
sc = spark.sparkContext

25/12/09 17:24:32 WARN Utils: Your hostname, cb4ae00-lcedt resolves to a loopback address: 127.0.1.1; using 10.110.47.100 instead (on interface eno1)
25/12/09 17:24:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/09 17:24:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### Load and Preprocess DataFrame

Load the WebMainBench [sample dataset](https://github.com/opendatalab/WebMainBench/blob/main/data/sample_dataset.jsonl).

In [7]:
import requests
import json

url = "https://raw.githubusercontent.com/opendatalab/WebMainBench/refs/heads/main/data/sample_dataset.jsonl"
response = requests.get(url)

data = []
for line in response.text.strip().split('\n'):
    if line:
        try:
            item = json.loads(line)
            record = {
                'html': item.get('html'),
                'groundtruth_content': item.get('content') or item.get('convert_main_content') or item.get('groundtruth_content'),
                'url': item.get('url')
            }
            if record['html'] and record['groundtruth_content']:
                data.append(record)
        except json.JSONDecodeError as e:
            print(f"Warning: Failed to parse line: {e}")
            continue

The sample only contains 4 records, so duplicating 25x to get 100 samples.

In [8]:
num_replicas = 25

pdf = pd.DataFrame(data)
pdf = pd.concat([pdf] * num_replicas, ignore_index=True)

print(pdf.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   html                 100 non-null    object
 1   groundtruth_content  100 non-null    object
 2   url                  100 non-null    object
dtypes: object(3)
memory usage: 2.5+ KB
None


In [9]:
df = spark.createDataFrame(pdf).repartition(2)

df.printSchema()
df.show(5, truncate=80)

root
 |-- html: string (nullable = true)
 |-- groundtruth_content: string (nullable = true)
 |-- url: string (nullable = true)



25/12/09 17:24:40 WARN TaskSetManager: Stage 0 contains a task of very large size (3883 KiB). The maximum recommended task size is 1000 KiB.


+--------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|                                                                            html|                                                                                       groundtruth_content|                                                                             url|
+--------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|<html class="avada-html-layout-wide" lang="en-US" prefix="og: http://ogp.me/n...|                          RCCI GROUP, INC. (“COMPANY” OR “WE” OR “OUR”) OPERATES THE TRIUMPH SERIES OF ..

## Inference using Spark DL API

Distributed inference using the PySpark [predict_batch_udf](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.functions.predict_batch_udf.html#pyspark.ml.functions.predict_batch_udf).

**Note 1:** 

Launching multiple vLLM v1 engines on the same machine will result in:
```python
AssertionError: Error in memory profiling. Initial free memory 44.5316162109375 GiB, current free memory 45.44561767578125 GiB. This happens when other processes sharing the same container release GPU memory while vLLM is profiling during initialization. To fix this, ensure consistent GPU memory allocation or isolate vLLM in its own container.
```
(Reference [here](https://github.com/vllm-project/vllm/blob/3c680f4a17057d7994af8fbb1dc8c2d98307c890/vllm/v1/worker/gpu_worker.py#L343)). As such, we disable v1 engine for inference.

**Note 2:** 

The settings for `gpu_memory_utilization` and `max_model_len` will heavily depend on your available GPU memory. This example requires loading two instances of the model (dataframe has two partitions) as well as additional space for activations and embeddings.

In [ ]:
def predict_batch_fn():
    import os
    import numpy as np
    from dripper.api import Dripper
    from pyspark import TaskContext

    os.environ["VLLM_USE_V1"] = "0"  # disable v1 to allow concurrent engines
    print(f"Initializing model on worker {TaskContext.get().partitionId()}")
    dripper = Dripper(
        config={
            'model_path': MODEL_PATH,
            'early_load': True,  # load upon worker initialization to ensure the model gets cached
            'vllm_kwargs': {
                'gpu_memory_utilization': 0.4,
                'max_model_len': 32768,
            }
        }
    )

    def predict(inputs):
        flattened = np.squeeze(inputs).tolist()
        outputs = dripper.process(flattened)
        return np.array([o.main_html for o in outputs])
    
    return predict

In [15]:
generate = predict_batch_udf(predict_batch_fn,
                             return_type=StringType(),
                             batch_size=50)

In [16]:
%%time
# first pass caches model/fn
preds = df.withColumn("processed_html", generate(struct("html")))
results = preds.collect()

25/12/09 17:26:01 WARN TaskSetManager: Stage 3 contains a task of very large size (3883 KiB). The maximum recommended task size is 1000 KiB.


CPU times: user 30.6 ms, sys: 36.6 ms, total: 67.1 ms
Wall time: 3min 24s


In [17]:
%%time
# first pass caches model/fn
preds = df.withColumn("processed_html", generate(struct("html")))
results = preds.collect()

25/12/09 17:29:35 WARN TaskSetManager: Stage 6 contains a task of very large size (3883 KiB). The maximum recommended task size is 1000 KiB.


CPU times: user 31.3 ms, sys: 25.3 ms, total: 56.6 ms
Wall time: 3min 10s
